In [ ]:
!pip install pandas numpy openai sentence-transformers datasets transformers[torch] accelerate>=1.1.0 seaborn hdbscan umap-learn nltk plotly optuna BERTopic "git+https://github.com/FelSiq/DBCV" evaluate trackio

In [ ]:
%load_ext cuml.accel

In [59]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import Literal
from openai import OpenAI
import random
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import safetensors
import umap
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
import textwrap
import gc
import re
import requests
import html
import re
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [60]:
pd.set_option('display.max_columns', None)

In [61]:
df_paragraph_level = pd.read_csv("filtered_paragraph_level_data.csv")

In [62]:
df_topics = pd.read_csv("data_subset_annotated_with_policy_targets.csv")

In [63]:
for column in [
    "canonical_target_en",
    "target_de",
    "target_fr",
    "target_it",
    "is_coherent_cluster",
    "is_specific_target",
    "is_stance_targetable",
    "target_abstain",
    "target_confidence",
]:
    if column not in df_topics.columns:
        if column in {"canonical_target_en", "target_de", "target_fr", "target_it"}:
            df_topics[column] = df_topics.get("c_labels")
        elif column == "target_confidence":
            df_topics[column] = "low"
        elif column == "target_abstain":
            df_topics[column] = False
        else:
            df_topics[column] = True

if "target_quality_pass" not in df_topics.columns:
    df_topics["target_quality_pass"] = (
        df_topics["is_coherent_cluster"].fillna(True)
        & df_topics["is_specific_target"].fillna(True)
        & df_topics["is_stance_targetable"].fillna(True)
        & ~df_topics["target_abstain"].fillna(False)
    )

speech_to_topic = df_topics.set_index('Speech_ID')['Topic'].to_dict()
speech_to_clabel = df_topics.set_index('Speech_ID')['canonical_target_en'].to_dict()
speech_to_target_de = df_topics.set_index('Speech_ID')['target_de'].to_dict()
speech_to_target_fr = df_topics.set_index('Speech_ID')['target_fr'].to_dict()
speech_to_target_it = df_topics.set_index('Speech_ID')['target_it'].to_dict()
speech_to_target_confidence = df_topics.set_index('Speech_ID')['target_confidence'].to_dict()
speech_to_target_quality_pass = df_topics.set_index('Speech_ID')['target_quality_pass'].to_dict()

In [64]:
def select_target_text(row):
    lang = str(row.get("lang", "DE")).upper().strip()
    if lang == "FR":
        return row.get("target_fr") or row.get("canonical_target_en")
    if lang == "IT":
        return row.get("target_it") or row.get("canonical_target_en")
    if lang == "EN":
        return row.get("canonical_target_en")
    return row.get("target_de") or row.get("canonical_target_en")


df_paragraph_level['Topic'] = df_paragraph_level['Speech_ID'].map(speech_to_topic)
df_paragraph_level['c_labels'] = df_paragraph_level['Speech_ID'].map(speech_to_clabel)
df_paragraph_level['canonical_target_en'] = df_paragraph_level['Speech_ID'].map(speech_to_clabel)
df_paragraph_level['target_de'] = df_paragraph_level['Speech_ID'].map(speech_to_target_de)
df_paragraph_level['target_fr'] = df_paragraph_level['Speech_ID'].map(speech_to_target_fr)
df_paragraph_level['target_it'] = df_paragraph_level['Speech_ID'].map(speech_to_target_it)
df_paragraph_level['target_confidence'] = df_paragraph_level['Speech_ID'].map(speech_to_target_confidence)
df_paragraph_level['target_quality_pass'] = df_paragraph_level['Speech_ID'].map(speech_to_target_quality_pass).fillna(False)
df_paragraph_level['target_text'] = df_paragraph_level.apply(select_target_text, axis=1)

df_paragraph_level.dropna(subset=['Topic', 'target_text'], inplace=True)
df_paragraph_level = df_paragraph_level[df_paragraph_level['Topic'] != -1].copy()
df_paragraph_level = df_paragraph_level[df_paragraph_level['target_quality_pass']].copy()
df_paragraph_level.reset_index(drop=True, inplace=True)

In [65]:
df_paragraph_level.head()

,Speech_ID,speaker_uid,speaker_first_name,speaker_last_name,speaker_fullname,party,party_name,gender,source_name,source_abbrev,meeting_uid,meeting_number,meeting_datetime,meeting_date,session_name,year,legislative_period_uid,Debate_ID,Bill_Number,Bill_Title_DE,Bill_Title_FR,Bill_Labels,Discussion_ID,Partition_ID,Paragraph_ID,name,role,language_of_text,TEXT_SPAN,lang,Affair_ID,Topic,c_labels,canonical_target_en,target_de,target_fr,target_it,target_confidence,target_quality_pass,target_text
0,83173,471.0,Felix,Gutzwiller,Gutzwiller Felix,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Gutzwiller Felix,NaN,DE,"Ich gehöre auch zur Minderheit, damit das klar...",DE,20040476,27.0,regulation of smoking areas in public establis...,regulation of smoking areas in public establis...,None,None,None,low,True,regulation of smoking areas in public establis...
1,83184,133.0,Ernst,Leuenberger,Leuenberger Ernst,SP,Sozialdemokratische Partei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,2,Leuenberger Ernst,NaN,DE,Ich will Ihnen in ein paar Sätzen Kenntnis von...,DE,20040476,114.0,protection against passive smoking,protection against passive smoking,None,None,None,low,True,protection against passive smoking
2,83184,133.0,Ernst,Leuenberger,Leuenberger Ernst,SP,Sozialdemokratische Partei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,3,Leuenberger Ernst,NaN,DE,Ich habe meine Schlüsse daraus gezogen und beg...,DE,20040476,114.0,protection against passive smoking,protection against passive smoking,None,None,None,low,True,protection against passive smoking
3,83187,408.0,This,Jenny,Jenny This,SVP,Schweizerische Volkspartei,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,8,Jenny This,NaN,DE,"Ich staune auch immer wieder, dass wir in der ...",DE,20040476,70.0,protection against passive smoking in public s...,protection against passive smoking in public s...,None,None,None,low,True,protection against passive smoking in public s...
4,83189,409.0,Hans,Hess,Hess Hans,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,3,Hess Hans,NaN,DE,Der Gesamtumsatz bei der Tabakproduktion beträ...,DE,20040476,70.0,protection against passive smoking in public s...,protection against passive smoking in public s...,None,None,None,low,True,protection against passive smoking in public s...


# NLI

In [66]:
gc.collect()

105

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)

In [ ]:
def get_stance(premise, hypothesis):
    encoded = tokenizer(
        premise,
        hypothesis,
        truncation="only_first",
        max_length=512,
        return_tensors="pt",
        padding=True,
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits[0]

    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    scores = {
        model.config.id2label[i].lower(): float(probs[i])
        for i in range(len(probs))
    }
    return scores

In [ ]:
get_stance(
"Die Schweiz sollte die EU-Mitgliedschaft anstreben.", "The Swiss government should pursue EU membership."
)

In [ ]:
SWISS_PARTY_COLORS = {
    'SVP': '#008751',
    'SP': '#E3001B',
    'FDP-Liberale': '#005691',
    'Grüne': '#84B414',
    'GLP': '#C3D000',
    'CVP': '#FF8C00',
    'Mitte': '#E65100',
    'BDP': '#FFD700',
    'Lega': '#002F6C',
    'MCG': '#F5B041',
    'AL': '#8E44AD'
}

In [ ]:
MULTILINGUAL_HYPOTHESES = {
    "DE": {
        "supports": lambda topic: f"Die sprechende Person unterstützt: {topic}.",
        "opposes": lambda topic: f"Die sprechende Person lehnt ab: {topic}.",
    },
    "FR": {
        "supports": lambda topic: f"La personne qui parle soutient : {topic}.",
        "opposes": lambda topic: f"La personne qui parle s'oppose à : {topic}.",
    },
    "IT": {
        "supports": lambda topic: f"La persona che parla sostiene: {topic}.",
        "opposes": lambda topic: f"La persona che parla si oppone a: {topic}.",
    },
    "EN": {
        "supports": lambda topic: f"The speaker supports: {topic}.",
        "opposes": lambda topic: f"The speaker opposes: {topic}.",
    },
}

In [ ]:
def compute_nli_scores_batched_streamlined(
    df,
    model,
    tokenizer,
    batch_size=128,
    device="cuda",
    text_col="TEXT_SPAN",
    topic_col="target_text",
    lang_col="lang",
):
    """
    Batched 2-hypothesis NLI scoring:
    premise = paragraph/window text
    hypothesis = speaker supports/opposes topic

    Returns entailment, contradiction, and net score for supports/opposes.
    """
    df_out = df.copy()
    keys = ["supports", "opposes"]

    # Label-safe indexing
    id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
    entailment_id = [i for i, lab in id2label.items() if lab == "entailment"][0]
    contradiction_id = [i for i, lab in id2label.items() if lab == "contradiction"][0]

    premises = []
    hypotheses = []
    flat_to_row_meta = []

    for idx, row in df_out.iterrows():
        premise = str(row[text_col])
        topic = str(row[topic_col])

        lang = str(row.get(lang_col, "DE")).upper().strip()
        if lang not in MULTILINGUAL_HYPOTHESES:
            lang = "DE"

        for key in keys:
            hypothesis = MULTILINGUAL_HYPOTHESES[lang][key](topic)

            premises.append(premise)
            hypotheses.append(hypothesis)
            flat_to_row_meta.append((idx, key))

    num_pairs = len(premises)
    entailments = np.zeros(num_pairs, dtype=np.float32)
    contradictions = np.zeros(num_pairs, dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for i in tqdm(
            range(0, num_pairs, batch_size),
            desc="Running 2-hypothesis NLI",
        ):
            batch_premises = premises[i : i + batch_size]
            batch_hypotheses = hypotheses[i : i + batch_size]

            encoded = tokenizer(
                batch_premises,
                batch_hypotheses,
                truncation="only_first",
                max_length=512,
                return_tensors="pt",
                padding=True,
            )

            encoded = {k: v.to(device) for k, v in encoded.items()}

            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

            n = len(batch_premises)
            entailments[i : i + n] = probs[:, entailment_id]
            contradictions[i : i + n] = probs[:, contradiction_id]

    for key in keys:
        df_out[f"nli_entail_{key}"] = np.nan
        df_out[f"nli_contra_{key}"] = np.nan
        df_out[f"nli_net_{key}"] = np.nan

    for i, (idx, key) in enumerate(flat_to_row_meta):
        entail = entailments[i]
        contra = contradictions[i]
        df_out.at[idx, f"nli_entail_{key}"] = entail
        df_out.at[idx, f"nli_contra_{key}"] = contra
        df_out.at[idx, f"nli_net_{key}"] = entail - contra

    df_out["nli_margin_support_minus_oppose"] = (
        df_out["nli_net_supports"] - df_out["nli_net_opposes"]
    )

    return df_out

In [ ]:
df_para_scored = compute_nli_scores_batched_streamlined(
    df=df_paragraph_level,
    model=model,
    tokenizer=tokenizer,
    batch_size=512,
    device=device,
)

In [ ]:
df_para_scored_clean = df_para_scored.dropna(subset=['target_text'])

In [ ]:
df_para_scored_clean

In [ ]:
def normalize_evidence_text(text):
    return " ".join(str(text).split())


def top_k_mean(series, top_k=3):
    values = series.dropna().sort_values(ascending=False)
    if values.empty:
        return np.nan
    return values.head(min(top_k, len(values))).mean()


def evidence_snippets(group, score_col, top_k=3):
    top_rows = group.nlargest(min(top_k, len(group)), score_col)
    snippets = []
    for _, row in top_rows.iterrows():
        snippets.append(
            f"[{row['Paragraph_ID']}] score={row[score_col]:.3f} :: {normalize_evidence_text(row['TEXT_SPAN'])}"
        )
    return "\n\n".join(snippets)


def aggregate_speech_stance(group, top_k=3):
    support_score = top_k_mean(group["nli_net_supports"], top_k=top_k)
    oppose_score = top_k_mean(group["nli_net_opposes"], top_k=top_k)
    margin = support_score - oppose_score

    best_support = group.loc[group["nli_net_supports"].idxmax()]
    best_oppose = group.loc[group["nli_net_opposes"].idxmax()]

    return pd.Series({
        "Affair_ID": group["Affair_ID"].iloc[0],
        "name": group["name"].iloc[0],
        "party": group["party"].iloc[0],
        "support_score": support_score,
        "oppose_score": oppose_score,
        "margin": margin,
        "best_support_evidence": normalize_evidence_text(best_support["TEXT_SPAN"]),
        "best_oppose_evidence": normalize_evidence_text(best_oppose["TEXT_SPAN"]),
        "support_topk_evidence": evidence_snippets(group, "nli_net_supports", top_k=top_k),
        "oppose_topk_evidence": evidence_snippets(group, "nli_net_opposes", top_k=top_k),
        "paragraph_count": len(group),
        "target_text": group["target_text"].iloc[0],
        "target_confidence": group["target_confidence"].iloc[0],
        "Topic": group["Topic"].iloc[0],
    })

In [ ]:
df_speech_scored = (
    df_para_scored_clean
    .groupby("Speech_ID", group_keys=False)
    .apply(lambda group: aggregate_speech_stance(group, top_k=3))
    .reset_index()
)

In [ ]:
df_speech_scored[((df_speech_scored['support_score'] > 0.7) & (df_speech_scored['margin'] >= 0.3)) | ((df_speech_scored['oppose_score'] > 0.7) & (df_speech_scored['margin'] >= 0.3))]

In [ ]:
df_speech_scored.iloc[13].to_dict()

## Advocacy Map Validation

In [ ]:
STANCE_DISPLAY_MIN_SPEECHES = 2
STANCE_DISPLAY_MIN_ABS_MARGIN = 0.2
TOP_TARGETS_TO_SHOW = 20

CONFIDENCE_TO_NUMERIC = {
    "low": 1,
    "medium": 2,
    "high": 3,
}


def confidence_to_numeric(series):
    return series.astype(str).str.lower().map(CONFIDENCE_TO_NUMERIC).fillna(1)


def confidence_from_numeric(value):
    if pd.isna(value):
        return "low"
    if value >= 2.5:
        return "high"
    if value >= 1.5:
        return "medium"
    return "low"


def display_target_label(row):
    target = row["canonical_target_en"]
    return f"T{int(row['Topic'])}: {target}"


df_cluster_validation = (
    df_topics[df_topics["Topic"] != -1]
    .groupby("Topic", as_index=False)
    .agg(
        canonical_target_en=("canonical_target_en", "first"),
        target_de=("target_de", "first"),
        target_fr=("target_fr", "first"),
        target_it=("target_it", "first"),
        target_confidence=("target_confidence", "first"),
        target_quality_pass=("target_quality_pass", "first"),
        target_abstain=("target_abstain", "first"),
        abstain_reason=("abstain_reason", "first"),
        cluster_size=("Speech_ID", "nunique"),
        sample_speech=("TEXT_SPAN", "first"),
        affair_id=("Affair_ID", "first"),
        lang_mix=("lang", lambda values: ", ".join(sorted(pd.Series(values).dropna().astype(str).unique().tolist()))),
    )
)
df_cluster_validation["target_confidence_num"] = confidence_to_numeric(df_cluster_validation["target_confidence"])
df_cluster_validation["target_label"] = df_cluster_validation.apply(display_target_label, axis=1)
df_cluster_validation = df_cluster_validation.sort_values(
    ["target_quality_pass", "target_confidence_num", "cluster_size"],
    ascending=[False, False, False],
).reset_index(drop=True)

df_advocacy_speech = df_speech_scored.merge(
    df_cluster_validation[["Topic", "canonical_target_en", "target_confidence", "target_quality_pass", "target_abstain", "target_label", "affair_id"]],
    on="Topic",
    how="left",
    suffixes=("", "_cluster"),
)
if "target_confidence_cluster" in df_advocacy_speech.columns:
    df_advocacy_speech["target_confidence"] = df_advocacy_speech["target_confidence"].fillna(
        df_advocacy_speech["target_confidence_cluster"]
    )
    df_advocacy_speech = df_advocacy_speech.drop(columns=["target_confidence_cluster"])
if "canonical_target_en_cluster" in df_advocacy_speech.columns:
    df_advocacy_speech["canonical_target_en"] = df_advocacy_speech["canonical_target_en"].fillna(
        df_advocacy_speech["canonical_target_en_cluster"]
    )
    df_advocacy_speech = df_advocacy_speech.drop(columns=["canonical_target_en_cluster"])
if "target_quality_pass_cluster" in df_advocacy_speech.columns:
    df_advocacy_speech["target_quality_pass"] = df_advocacy_speech["target_quality_pass"].fillna(
        df_advocacy_speech["target_quality_pass_cluster"]
    )
    df_advocacy_speech = df_advocacy_speech.drop(columns=["target_quality_pass_cluster"])
if "target_abstain_cluster" in df_advocacy_speech.columns:
    df_advocacy_speech["target_abstain"] = df_advocacy_speech["target_abstain"].fillna(
        df_advocacy_speech["target_abstain_cluster"]
    )
    df_advocacy_speech = df_advocacy_speech.drop(columns=["target_abstain_cluster"])
if "target_label_cluster" in df_advocacy_speech.columns:
    df_advocacy_speech["target_label"] = df_advocacy_speech["target_label"].fillna(
        df_advocacy_speech["target_label_cluster"]
    )
    df_advocacy_speech = df_advocacy_speech.drop(columns=["target_label_cluster"])
df_advocacy_speech["stance_score"] = df_advocacy_speech["support_score"] - df_advocacy_speech["oppose_score"]
df_advocacy_speech["abs_stance_score"] = df_advocacy_speech["stance_score"].abs()
df_advocacy_speech["displayable_cell"] = (
    df_advocacy_speech["target_quality_pass"].fillna(False)
    & (df_advocacy_speech["abs_stance_score"] >= STANCE_DISPLAY_MIN_ABS_MARGIN)
)

df_advocacy_mp_target = (
    df_advocacy_speech.groupby(["name", "party", "Topic", "canonical_target_en", "target_label"], as_index=False)
    .agg(
        speech_count=("Speech_ID", "nunique"),
        mean_support_score=("support_score", "mean"),
        mean_oppose_score=("oppose_score", "mean"),
        mean_margin=("margin", "mean"),
        mean_stance_score=("stance_score", "mean"),
        mean_abs_stance_score=("abs_stance_score", "mean"),
        target_confidence_num=("target_confidence", lambda values: confidence_to_numeric(pd.Series(values)).mean()),
        best_support_evidence=("best_support_evidence", "first"),
        best_oppose_evidence=("best_oppose_evidence", "first"),
        support_topk_evidence=("support_topk_evidence", "first"),
        oppose_topk_evidence=("oppose_topk_evidence", "first"),
        affair_id=("affair_id", "first")
    )
)
df_advocacy_mp_target["target_confidence"] = df_advocacy_mp_target["target_confidence_num"].map(confidence_from_numeric)
df_advocacy_mp_target["displayable_cell"] = (
    (df_advocacy_mp_target["speech_count"] >= STANCE_DISPLAY_MIN_SPEECHES)
    & (df_advocacy_mp_target["mean_abs_stance_score"] >= STANCE_DISPLAY_MIN_ABS_MARGIN)
)
df_advocacy_mp_target = df_advocacy_mp_target.sort_values(
    ["speech_count", "mean_abs_stance_score"],
    ascending=[False, False],
).reset_index(drop=True)

bill_metadata = (
    df_para_scored_clean.groupby("Speech_ID", as_index=False)
    .agg(
        Bill_Number=("Bill_Number", "first"),
        Bill_Title_DE=("Bill_Title_DE", "first"),
        Bill_Title_FR=("Bill_Title_FR", "first"),
        year=("year", "first"),
    )
)
df_advocacy_speech = df_advocacy_speech.merge(bill_metadata, on="Speech_ID", how="left")

df_bill_case_studies = (
    df_advocacy_speech[df_advocacy_speech["target_quality_pass"].fillna(False)]
    .groupby(["Bill_Number", "Bill_Title_DE", "Bill_Title_FR"], as_index=False)
    .agg(
        speech_count=("Speech_ID", "nunique"),
        mp_count=("name", "nunique"),
        topic_count=("Topic", "nunique"),
        mean_abs_stance=("abs_stance_score", "mean"),
        year=("year", "first"),
        affair_id=("affair_id", "first")
    )
)
df_bill_case_studies = df_bill_case_studies.sort_values(
    ["topic_count", "speech_count", "mp_count"],
    ascending=[False, False, False],
).reset_index(drop=True)

In [ ]:
cluster_quality_columns = [
    "Topic",
    "target_label",
    "cluster_size",
    "lang_mix",
    "target_confidence",
    "target_quality_pass",
    "target_abstain",
    "abstain_reason",
    "sample_speech",
]

df_cluster_validation[cluster_quality_columns].head(25)

In [ ]:
df_bill_case_studies.sort_values('speech_count', ascending=False).head(25)

In [ ]:
selected_bill_number = df_bill_case_studies.sort_values('speech_count', ascending=False).iloc[2]["Bill_Number"]

bill_case = df_advocacy_speech[df_advocacy_speech["Bill_Number"] == selected_bill_number].copy()
bill_case = bill_case[bill_case["target_quality_pass"].fillna(False)]
bill_title_de = bill_case["Bill_Title_DE"].dropna().iloc[0] if bill_case["Bill_Title_DE"].notna().any() else ""
bill_title_fr = bill_case["Bill_Title_FR"].dropna().iloc[0] if bill_case["Bill_Title_FR"].notna().any() else ""

bill_topic_summary = (
    bill_case.groupby(["Topic", "target_label", "target_confidence"], as_index=False)
    .agg(
        speech_count=("Speech_ID", "nunique"),
        mp_count=("name", "nunique"),
        mean_abs_stance=("abs_stance_score", "mean"),
        sample_support=("best_support_evidence", "first"),
        sample_oppose=("best_oppose_evidence", "first"),
    )
    .sort_values(["speech_count", "mp_count"], ascending=[False, False])
)

bill_topic_summary

In [ ]:
fig_bill_topics = px.bar(
    bill_topic_summary,
    x="speech_count",
    y="target_label",
    color="target_confidence",
    orientation="h",
    hover_data=["mp_count", "mean_abs_stance", "sample_support", "sample_oppose"],
    title=f"Subtopics identified inside bill {selected_bill_number}",
    labels={
        "speech_count": "Speeches assigned to subtopic",
        "target_label": "Refined subtopic",
    },
)
fig_bill_topics.update_layout(height=max(450, 35 * len(bill_topic_summary)))
fig_bill_topics.show()

print(f"Bill {selected_bill_number} DE: {bill_title_de}")
print(f"Bill {selected_bill_number} FR: {bill_title_fr}")

In [ ]:
selected_topic = bill_topic_summary.iloc[0]["Topic"]

bill_topic_mp_summary = (
    bill_case[bill_case["Topic"] == selected_topic]
    .groupby(["name", "party", "Topic", "target_label"], as_index=False)
    .agg(
        speech_count=("Speech_ID", "nunique"),
        mean_support_score=("support_score", "mean"),
        mean_oppose_score=("oppose_score", "mean"),
        mean_margin=("margin", "mean"),
        mean_stance_score=("stance_score", "mean"),
        best_support_evidence=("best_support_evidence", "first"),
        best_oppose_evidence=("best_oppose_evidence", "first"),
    )
    .sort_values("mean_stance_score")
)

fig_bill_opposition = px.bar(
    bill_topic_mp_summary,
    x="mean_stance_score",
    y="name",
    color="party",
    orientation="h",
    hover_data=["speech_count", "best_support_evidence", "best_oppose_evidence"],
    title=f"Who opposed the most on {bill_topic_summary.iloc[0]['target_label']}",
    labels={
        "mean_stance_score": "Mean advocacy score",
        "name": "MP",
    },
)
fig_bill_opposition.add_vline(x=0, line_dash="dash", line_color="black")
fig_bill_opposition.update_layout(height=max(500, 22 * len(bill_topic_mp_summary)))
fig_bill_opposition.show()

bill_topic_mp_summary.head(20)

In [ ]:
selected_mp_topic = bill_topic_mp_summary.iloc[0]

evidence_drilldown = df_para_scored_clean[
    (df_para_scored_clean["Bill_Number"] == selected_bill_number)
    & (df_para_scored_clean["Topic"] == selected_topic)
    & (df_para_scored_clean["name"] == selected_mp_topic["name"])
].copy()
evidence_drilldown = evidence_drilldown.sort_values(
    ["nli_margin_support_minus_oppose", "nli_net_supports"],
    ascending=[False, False],
)

evidence_drilldown[[
    "Speech_ID",
    "Paragraph_ID",
    "lang",
    "target_text",
    "nli_net_supports",
    "nli_net_opposes",
    "nli_margin_support_minus_oppose",
    "TEXT_SPAN",
]].head(25)

In [ ]:
bill_cluster_review_sample = (
    bill_case[["Bill_Number", "Topic", "target_label", "target_confidence"]]
    .drop_duplicates(subset=["Topic"])
    .groupby("Topic", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)
bill_cluster_review_sample["human_cluster_coherent"] = pd.NA
bill_cluster_review_sample["human_target_specific"] = pd.NA
bill_cluster_review_sample["human_target_stanceable"] = pd.NA

bill_stance_review_sample = bill_case.copy()
bill_stance_review_sample["score_band"] = pd.cut(
    bill_stance_review_sample["stance_score"],
    bins=[-np.inf, -0.5, -0.2, 0.2, 0.5, np.inf],
    labels=["strong_oppose", "lean_oppose", "ambiguous", "lean_support", "strong_support"],
)
bill_stance_review_sample = (
    bill_stance_review_sample.groupby("score_band", group_keys=False)
    .apply(lambda frame: frame.sample(min(len(frame), 4), random_state=42))
    .reset_index(drop=True)
)
bill_stance_review_sample["human_stance_label"] = pd.NA
bill_stance_review_sample["human_on_target"] = pd.NA

bill_cluster_review_sample[[
    "Bill_Number",
    "Topic",
    "target_label",
    "target_confidence",
    "human_cluster_coherent",
    "human_target_specific",
    "human_target_stanceable",
]]

bill_stance_review_sample[[
    "Speech_ID",
    "name",
    "party",
    "target_label",
    "stance_score",
    "score_band",
    "target_confidence",
    "best_support_evidence",
    "best_oppose_evidence",
    "human_stance_label",
    "human_on_target",
]]